# Hugging Face Transformers with Python

This notebook shows a minimal way to run a local Hugging Face `transformers` text-generation pipeline.


In [2]:
%pip install -q transformers torch python-dotenv


Note: you may need to restart the kernel to use updated packages.


## Model setup

The notebook reads `HF_MODEL` from the repo-local `.env` file if present.

Example:

```text
HF_MODEL=gpt2
```

The first run downloads model weights locally, so it can take longer than later runs.


In [3]:
import os
from pathlib import Path

from dotenv import load_dotenv

load_dotenv(Path.cwd() / '.env', override=False)

MODEL = os.environ.get('HF_MODEL', 'gpt2')

print(f'Using model: {MODEL}')


Using model: gpt2


In [4]:
from transformers import pipeline, set_seed

set_seed(42)

_generators = {}

def get_generator(model: str = MODEL):
    if model not in _generators:
        _generators[model] = pipeline('text-generation', model=model)
    return _generators[model]


c:\OneDrive\Documents\github_repos\ml_coding\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Reusable helper

This helper wraps the `transformers` text-generation pipeline and strips the prompt from the returned text.


In [5]:
def ask_huggingface(prompt: str, model: str = MODEL, max_tokens: int = 200) -> str:
    generator = get_generator(model)
    try:
        result = generator(
            prompt,
            max_new_tokens=max_tokens,
            num_return_sequences=1,
            truncation=True,
        )
        generated_text = result[0]['generated_text']
        if generated_text.startswith(prompt):
            return generated_text[len(prompt):].strip()
        return generated_text.strip()
    except Exception as exc:
        return f'Unexpected error: {exc}'


In [6]:
generator = get_generator(MODEL)
print(generator("Hello, I'm a language model,", max_length=30, num_return_sequences=5))


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 839.56it/s, Materializing param=transformer.wte.weight]             
Passing `generation_config` together with generation-related arguments=({'max_length', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[{'generated_text': "Hello, I'm a language model, I'm a language model. In my mind, I'm doing the same thing as you. All these different people are thinking about the same thing.\n\nI'm not talking about what the computer program does. I'm talking about what I'm thinking about. I'm thinking about what my body is thinking about. I'm thinking about what I'm making it. I'm thinking about what I'm making it. What my body is doing. I'm thinking about what my body is thinking about. What's my body doing? What's my body doing? What's my body doing?\n\nLet's talk about this, I'm not a computer programmer. I'm not a software developer. I'm not a computer programmer. I'm not a language model. I'm not a language model. My body is thinking about what I'm doing. What is my body doing? What's my body doing? What's my body doing?\n\nNow, I'm not saying that this is a good thing. But if you want to get better at programming, you can get better at writing. You can get better at being human. You can get

## Test prompts

These are the two prompts you asked to try.


In [7]:
test_prompts = [
    'Who is Tom Cruise\'s mother?',
    'Who is Mary Lee Pfeiffer\'s son?',
]

for prompt in test_prompts:
    print(f'Prompt: {prompt}')
    print(ask_huggingface(prompt, max_tokens=30))
    print('-' * 80)


Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'num_return_sequences'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Prompt: Who is Tom Cruise's mother?


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Tom Cruise's mother is a huge fan of the show. She says, "I'm a big fan of her, but I don't know whether
--------------------------------------------------------------------------------
Prompt: Who is Mary Lee Pfeiffer's son?
The name Mary Lee Pfeiffer has been changed.

The word "Pfeiffer" was added to the name of
--------------------------------------------------------------------------------


In [8]:
test_prompts = [
    'Who is Tom Cruise\'s mother?',
    'Who is Mary Lee Pfeiffer\'s son?',
]

for prompt in test_prompts:
    print(f'Prompt: {prompt}')
    print(ask_huggingface(prompt, model='meta-llama/Llama-3.2-3B-Instruct', max_tokens=30))
    print('-' * 80)

Prompt: Who is Tom Cruise's mother?


Loading weights: 100%|██████████| 254/254 [00:00<00:00, 841.26it/s, Materializing param=model.norm.weight]                              
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
Both `max_new_tokens` (=30) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Mary Lee Pfeiffer?
I think you may have made a mistake. Tom Cruise's mother is actually Mary Lee Pfeiffer, but I think
--------------------------------------------------------------------------------
Prompt: Who is Mary Lee Pfeiffer's son?
I'm unable to find this information.  Can you provide some guidance or possible sources?
I am trying to research Mary Lee Pfeiffer's
--------------------------------------------------------------------------------
